# JanoGPT Training on Google Colab (TPU)

This notebook trains a GPT-2 model using JanoGPT on Google Colab's **TPU v5e-1**.

**What this notebook does:**
1. Downloads JanoGPT code from GitHub
2. Installs TPU-optimized JAX
3. Sets up WandB for tracking
4. Downloads OpenWebText dataset
5. Trains GPT-2 124M for 100 steps (demo)
6. Saves checkpoints to Google Drive

**Memory configuration:**
- Conservative batch sizing for 16GB TPU (v5e)
- micro_batch=4, accumulation=128 → ~524K tokens/step
- Estimated memory: ~5.1GB (fits comfortably in 16GB)

## 1. Setup Environment & Check TPU

In [1]:
# Check TPU availability
import os

try:
    import jax
    print(f"JAX version: {jax.__version__}")
    print(f"Devices: {jax.devices()}")
    print(f"Device count: {jax.local_device_count()}")
    print(f"Device type: {jax.devices()[0].platform}")

    if jax.devices()[0].platform == 'tpu':
        print("\n✓ TPU detected!")
    else:
        print(f"\n⚠️  Warning: Running on {jax.devices()[0].platform}, not TPU")
        print("   Go to Runtime → Change runtime type → TPU v5 litepod")
except Exception as e:
    print(f"⚠️  JAX not installed or error: {e}")
    print("Will install in next cell")

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


JAX version: 0.7.2
Devices: [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0)]
Device count: 1
Device type: tpu

✓ TPU detected!


In [2]:
# Clone JanoGPT repository
!git clone https://github.com/hhe0u0/janogpt.git
%cd janogpt
!pip install -e .

Cloning into 'janogpt'...
remote: Enumerating objects: 332, done.
remote: Counting objects: 100% (332/332), done.
remote: Compressing objects: 100% (176/176), done.
remote: Total 332 (delta 187), reused 277 (delta 145), pack-reused 0 (from 0)
Receiving objects: 100% (332/332), 186.08 KiB | 7.16 MiB/s, done.
Resolving deltas: 100% (187/187), done.
/content/janogpt
Obtaining file:///content/janogpt
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.2/27.2 MB 109.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.2/212.2 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 468.4/468.4 kB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 6.1 MB/s eta 0:0

In [3]:
# Verify TPU setup
import jax
import jax.numpy as jnp

print(f"JAX version: {jax.__version__}")
print(f"JAX devices: {jax.devices()}")
print(f"Device count: {jax.local_device_count()}")
print(f"Device type: {jax.devices()[0].platform}")

# Test TPU with simple operation
x = jnp.ones((1000, 1000))
y = jnp.dot(x, x)
print(f"\n✓ TPU test successful: {y.shape}")

if jax.devices()[0].platform != 'tpu':
    print("\n⚠️  WARNING: Not running on TPU!")
    print("   Runtime → Change runtime type → TPU v5 litepod")

JAX version: 0.7.2
JAX devices: [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0)]
Device count: 1
Device type: tpu

✓ TPU test successful: (1000, 1000)


## 2. Mount Google Drive

We'll save checkpoints to Google Drive so they persist after the Colab session ends.

In [4]:
from google.colab import drive
drive.mount('/content/drive')

# Create checkpoint directory in Google Drive
import os
drive_checkpoint_dir = "/content/drive/MyDrive/colab/janogpt_checkpoints"
os.makedirs(drive_checkpoint_dir, exist_ok=True)
print(f"✓ Checkpoints will be saved to: {drive_checkpoint_dir}")

Mounted at /content/drive
✓ Checkpoints will be saved to: /content/drive/MyDrive/colab/janogpt_checkpoints


## 3. Setup WandB (Optional)

WandB tracks your training metrics.

**Options:**
- **Option A:** Login interactively (paste your key when prompted)
- **Option B:** Use Colab secrets (add WANDB_API_KEY in Secrets)
- **Option C:** Skip WandB (set `wandb_enabled = False` below)

In [5]:
import os

# Configuration
wandb_enabled = True  # Set to False to disable WandB

if wandb_enabled:
    try:
        # Try to get WandB key from Colab secrets
        from google.colab import userdata
        wandb_key = userdata.get('WANDB_API_KEY')
        os.environ["WANDB_API_KEY"] = wandb_key
        print("✓ Loaded WandB API key from Colab secrets")
    except:
        print("⚠️  WandB secret not found. Interactive login...")
        import wandb
        wandb.login()
else:
    print("ℹ️  WandB tracking disabled")

✓ Loaded WandB API key from Colab secrets


## 4. Download Training Data to Google Drive

We'll download the OpenWebText dataset (~20GB) to Google Drive so it persists between sessions.

**First time:** Downloads dataset to Drive (~10 minutes)
**Future sessions:** Uses existing dataset (instant!)

**Dataset will be saved to:** `/content/drive/MyDrive/colab/janogpt_datasets/openwebtext/`

In [6]:
# Check if dataset already exists in Google Drive
from pathlib import Path
import kagglehub
import os

# Google Drive dataset location
drive_data_dir = Path("/content/drive/MyDrive/colab/janogpt_datasets/openwebtext")
drive_train_bin = drive_data_dir / "train.bin"
drive_val_bin = drive_data_dir / "val.bin"

if drive_train_bin.exists() and drive_val_bin.exists():
    print(f"✓ Dataset already exists in Google Drive!")
    print(f"  Location: {drive_data_dir}")
    print(f"  train.bin: {drive_train_bin.stat().st_size / 1e9:.2f} GB")
    print(f"  val.bin: {drive_val_bin.stat().st_size / 1e6:.2f} MB")
    print("\n✓ Skipping download (using existing dataset)")
    dataset_ready = True
else:
    print(f"Dataset not found in Google Drive")
    print(f"Will download to: {drive_data_dir}")
    print("\n⏳ This will take ~10 minutes (first time only)")

    try:
      # Download the latest version of a dataset to a specific directory
      path = kagglehub.dataset_download(
          "windmaple/openwebtext-gpt2",
          output_dir=drive_data_dir
      )
      dataset_ready = True
    except Exception as e:
      print(f"⚠️  Error downloading dataset: {e}")
      dataset_ready = False

✓ Dataset already exists in Google Drive!
  Location: /content/drive/MyDrive/colab/janogpt_datasets/openwebtext
  train.bin: 18.07 GB
  val.bin: 8.87 MB

✓ Skipping download (using existing dataset)


In [7]:
# Check if dataset already exists in Google Drive
from pathlib import Path
import os

# Google Drive dataset location
drive_data_dir = Path("/content/drive/MyDrive/colab/janogpt_datasets/openwebtext")
drive_train_bin = drive_data_dir / "train.bin"
drive_val_bin = drive_data_dir / "val.bin"

if drive_train_bin.exists() and drive_val_bin.exists():
    print(f"✓ Dataset already exists in Google Drive!")
    print(f"  Location: {drive_data_dir}")
    print(f"  train.bin: {drive_train_bin.stat().st_size / 1e9:.2f} GB")
    print(f"  val.bin: {drive_val_bin.stat().st_size / 1e6:.2f} MB")
    print("\n✓ Skipping download (using existing dataset)")
    dataset_ready = True
else:
    print(f"Dataset not found in Google Drive")
    print(f"Will download to: {drive_data_dir}")
    print("\n⏳ This will take ~10 minutes (first time only)")
    dataset_ready = False

✓ Dataset already exists in Google Drive!
  Location: /content/drive/MyDrive/colab/janogpt_datasets/openwebtext
  train.bin: 18.07 GB
  val.bin: 8.87 MB

✓ Skipping download (using existing dataset)


In [8]:
# Create symbolic links from Drive to local working directory
from pathlib import Path
import os

# Local working directory
work_data_dir = Path("data/openwebtext")
work_data_dir.mkdir(parents=True, exist_ok=True)

# Google Drive dataset location
drive_data_dir = Path("/content/drive/MyDrive/colab/janogpt_datasets/openwebtext")
drive_train_bin = drive_data_dir / "train.bin"
drive_val_bin = drive_data_dir / "val.bin"

if drive_train_bin.exists() and drive_val_bin.exists():
    # Create symbolic links from Drive
    if not (work_data_dir / "train.bin").exists():
        os.symlink(drive_train_bin, work_data_dir / "train.bin")
    if not (work_data_dir / "val.bin").exists():
        os.symlink(drive_val_bin, work_data_dir / "val.bin")

    print(f"✓ Data linked from Google Drive to {work_data_dir}")
    print(f"  train.bin -> {drive_train_bin}")
    print(f"  val.bin -> {drive_val_bin}")
    print("\n✓ Ready to train!")
else:
    print("⚠️  Dataset not found in Drive. Creating dummy data...")
    import numpy as np

    # Create small dummy dataset for testing
    dummy_train = np.random.randint(0, 50257, size=5_000_000, dtype=np.uint16)
    dummy_val = np.random.randint(0, 50257, size=500_000, dtype=np.uint16)

    dummy_train.tofile(work_data_dir / "train.bin")
    dummy_val.tofile(work_data_dir / "val.bin")

    print(f"✓ Created dummy dataset at {work_data_dir}")
    print("  (For testing only - download real dataset for actual training)")

✓ Data linked from Google Drive to data/openwebtext
  train.bin -> /content/drive/MyDrive/colab/janogpt_datasets/openwebtext/train.bin
  val.bin -> /content/drive/MyDrive/colab/janogpt_datasets/openwebtext/val.bin

✓ Ready to train!


## 5. Create Training Configuration

We'll create a TPU-optimized config for GPT-2 124M with conservative memory settings.

**Memory Configuration (Conservative for 16GB TPU):**
- Base overhead: `40 × 0.124B params = ~5GB` (model + gradients + optimizer)
- Activations: `~34MB per sequence` (empirically measured for GPT-2 124M, seq_len=1024)
- **micro_batch=4:** Uses ~5.1GB total, leaves ~11GB headroom
- **gradient_accumulation=128:** Maintains ~524K tokens/step target

**Why conservative?**
- JAX/XLA have additional overhead beyond model memory
- Colab TPUs may have slightly less available than nominal 16GB
- Safer to start small and scale up if stable

In [ ]:
import json
from pathlib import Path

# Get number of TPU cores
import jax
num_devices = jax.local_device_count()
print(f"Detected {num_devices} TPU core(s)")

micro_batch_size = 1  # Conservative for 16GB TPU
gradient_accumulation_steps = 512 // micro_batch_size // num_devices  # Target ~0.5M tokens
effective_tokens = micro_batch_size * gradient_accumulation_steps * num_devices * 1024

print(f"\nMemory-conservative batch configuration:")
print(f"  micro_batch_size: {micro_batch_size} (per core)")
print(f"  gradient_accumulation: {gradient_accumulation_steps}")
print(f"  num_devices: {num_devices}")
print(f"  Effective batch: {effective_tokens:,} tokens per step (~{effective_tokens/1e6:.2f}M)")
print(f"\nEstimated memory usage:")
print(f"  Base (model + opt): ~5.0GB")
print(f"  Activations (micro={micro_batch_size}): ~{micro_batch_size * 34}MB")
print(f"  Total estimated: ~{5.0 + micro_batch_size * 34 / 1000:.2f}GB")
print(f"  Should fit in 16GB TPU with ~10GB headroom")

config = {
    "_comment": f"GPT-2 124M training config for Colab TPU ({num_devices} core(s))",

    "model": {
        "dropout_prob": 0.1,
        "num_blocks": 12,
        "emb_dim": 768,
        "num_heads": 12,
        "seq_len": 1024,
        "epsilon": 1e-6,
        "voc_size": 50304
    },

    "optimizer": {
        "learning_rate": 6e-4,
        "min_learning_rate": 6e-5,
        "warmup_steps": 100,
        "beta1": 0.9,
        "beta2": 0.95,
        "grad_clip": 1.0,
        "weight_decay": 0.1
    },

    "training": {
        "max_steps": 100,
        "micro_batch_size": micro_batch_size,
        "gradient_accumulation_steps": gradient_accumulation_steps,
        "_memory_comment": f"Base 5GB + micro×34MB = {5.0 + micro_batch_size * 34 / 1000:.2f}GB",
        "_effective_batch_comment": f"{micro_batch_size} × {gradient_accumulation_steps} × {num_devices} × 1024 = {effective_tokens:,} tokens",
        "seed": 42
    },

    "data": {
        "data_dir": "data/openwebtext",
        "train_file": "train.bin",
        "val_file": "val.bin"
    },

    "logging": {
        "eval_interval": 20,
        "eval_iters": 50,
        "log_interval": 10
    },

    "checkpointing": {
        "save_interval": 50,
        "output_dir": "output_colab_tpu",
        "resume_from_checkpoint": None
    },

    "wandb": {
        "enabled": wandb_enabled,
        "project": "janogpt-colab-tpu",
        "run_name": f"gpt2-124m-tpu{num_devices}-100steps",
        "tags": ["colab", "tpu", "gpt2", "100-steps"]
    }
}

# Save config
config_dir = Path("configs")
config_dir.mkdir(exist_ok=True)
config_path = config_dir / "train_colab_tpu.json"

with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

print(f"\n✓ Config saved to {config_path}")
print("\nConfiguration Summary:")
print(f"  Model: GPT-2 124M ({config['model']['num_blocks']} layers, {config['model']['emb_dim']} dim)")
print(f"  Training: {config['training']['max_steps']} steps")
print(f"  Effective batch: {effective_tokens:,} tokens per step")
print(f"  TPU cores: {num_devices}")
print(f"  Checkpoints: Every {config['checkpointing']['save_interval']} steps")
print(f"  WandB: {'Enabled' if config['wandb']['enabled'] else 'Disabled'}")

Detected 1 TPU cores
Effective batch size: 1,048,576 tokens per step
  micro_batch_size: 2
  gradient_accumulation: 512
  num_devices: 1

✓ Config saved to configs/train_colab_tpu_1k.json

Configuration Summary:
  Model: GPT-2 124M (12 layers, 768 dim)
  Training: 100 steps
  Effective batch: 1,048,576 tokens per step
  TPU cores: 1
  Checkpoints: Every 50 steps
  WandB: Enabled


## 6. Train the Model

Now we'll train for 100 steps on TPU.

**Expected TPU v5e-1 performance:**
- Tokens/sec: ~8,000-16,000
- Steps/sec: ~2-4
- Memory usage: ~5.1GB per TPU core
- Time for 100 steps: ~5-10 minutes

**Checkpoints will be saved to:** `output_colab_tpu/checkpoints/`

In [ ]:
# Start training (inline to preserve TPU session)
import sys
sys.path.insert(0, '.')

from janogpt import GPT, Config, Trainer
from janogpt.logger import ConsoleLogger, DatasetEvaluator, MultiLogger, WandBLogger
from janogpt.utils import FileDataLoader

# Load config
config = Config.from_json(config_path)

print("=" * 80)
print("JanoGPT Training on Colab TPU")
print("=" * 80)
print(f"Config: {config_path}")
print(f"Devices: {jax.local_device_count()} {jax.devices()[0].platform}")
print("=" * 80)

# Calculate effective batch
num_devices = jax.local_device_count()
effective_batch = config.micro_batch_size * config.gradient_accumulation_steps * num_devices

# Create data loaders
print(f"\nLoading data from {config.data_dir}")
train_loader = FileDataLoader(
    data_dir=config.data_dir,
    batch_size=effective_batch,
    seq_len=config.seq_len,
    split="train",
    seed=config.seed,
)
val_loader = FileDataLoader(
    data_dir=config.data_dir,
    batch_size=effective_batch,
    seq_len=config.seq_len,
    split="val",
    seed=config.seed + 1,
)

# Create model
model = GPT(config)

# Create evaluators
evaluators = [DatasetEvaluator(val_loader, config.eval_iters, "val", model=model)]

# Create loggers
console = ConsoleLogger()
wandb_logger = WandBLogger(enabled=config.wandb_log)
logger = MultiLogger([console, wandb_logger])

# Create trainer
trainer = Trainer(
    model=model, config=config, evaluators=evaluators, logger=logger, seed=config.seed
)

# Train
print("\nStarting training...")
try:
    trainer.train(train_loader, verbose_first_step=True)
except KeyboardInterrupt:
    print("\n\nTraining interrupted by user")
    trainer.save_checkpoint(int(trainer.state.step))

print("\n✓ Training complete!")

JanoGPT Training on TPU v6e-1
Config: configs/train_kaggle_tpu.json
Devices: 1 tpu

Loading data from data/openwebtext
Loaded train data: 9,035,582,489 tokens
Loaded val data: 4,434,606 tokens
Detected 1 TPU device(s)
Effective batch: 1024 seqs = 1049K tokens per step
Initializing model...
Initializing model parameters...
✓ Model initialized: 124.48M parameters (8.9s)
Creating optimizer...
⚠️  Reduced warmup_steps from 100 to 99 (max_steps=100)
✓ Console logger initialized (model: 124.48M params)


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: cipher0u0 (cipher0u0-amazon) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


✓ WandB initialized: https://wandb.ai/cipher0u0-amazon/janogpt-colab-tpu/runs/qfr4syrv
Setting up JIT compilation for single device...
✓ JIT compilation configured (will compile on first step)

Starting training...

Starting training...
[step 1] Starting first train step...
  - Effective batch: 1024 seqs, 1049K tokens
  - Running verbose (non-JIT) version to show progress...
  - Processing 512 gradient accumulation steps...
    [51/512] 10% complete - loss: 11.2772
    [102/512] 20% complete - loss: 11.2653
    [153/512] 30% complete - loss: 11.2611
    [204/512] 40% complete - loss: 11.2993
    [255/512] 50% complete - loss: 11.3262
    [306/512] 60% complete - loss: 11.2624
    [357/512] 70% complete - loss: 11.2996
    [408/512] 80% complete - loss: 11.1990
    [459/512] 90% complete - loss: 11.2680
    [510/512] 100% complete - loss: 11.2561
  ✓ All micro-batches processed
  - Applying gradients...
⚠️  Reduced warmup_steps from 100 to 99 (max_steps=100)
[step 1] ✓ First step comple

AttributeError: 'Trainer' object has no attribute 'state'

## 7. Upload Checkpoints to Google Drive

Copy checkpoints from local storage to Google Drive so they persist after the session ends.

In [ ]:
import shutil
from pathlib import Path
import time

# Source and destination
local_checkpoint_dir = Path("output_colab_tpu/checkpoints")
drive_checkpoint_dir = Path("/content/drive/MyDrive/colab/janogpt_checkpoints/colab_tpu")
drive_checkpoint_dir.mkdir(parents=True, exist_ok=True)

if local_checkpoint_dir.exists():
    checkpoints = sorted(local_checkpoint_dir.glob("step_*"))
    print(f"Found {len(checkpoints)} checkpoint(s) to upload")

    for ckpt in checkpoints:
        dest = drive_checkpoint_dir / ckpt.name

        if dest.exists():
            print(f"  {ckpt.name}: Already in Drive, skipping")
        else:
            print(f"  {ckpt.name}: Uploading...", end="", flush=True)
            start = time.time()
            shutil.copytree(ckpt, dest)
            elapsed = time.time() - start

            # Check size
            size_mb = sum(f.stat().st_size for f in dest.rglob("*") if f.is_file()) / 1e6
            print(f" ✓ ({size_mb:.1f} MB, {elapsed:.1f}s)")

    print(f"\n✓ All checkpoints uploaded to: {drive_checkpoint_dir}")
    print(f"  Total checkpoints: {len(checkpoints)}")
else:
    print("⚠️  No checkpoints found to upload")

## 8. Verify Checkpoints in Google Drive

In [ ]:
# List checkpoints in Google Drive
drive_checkpoint_dir = Path("/content/drive/MyDrive/colab/janogpt_checkpoints/colab_tpu")

if drive_checkpoint_dir.exists():
    checkpoints = sorted(drive_checkpoint_dir.glob("step_*"))
    print(f"✓ Checkpoints in Google Drive ({len(checkpoints)}):")

    for ckpt in checkpoints:
        size_mb = sum(f.stat().st_size for f in ckpt.rglob("*") if f.is_file()) / 1e6
        print(f"  {ckpt.name}: {size_mb:.1f} MB")

    if checkpoints:
        latest = checkpoints[-1]
        print(f"\n✓ Latest checkpoint: {latest.name}")
        print(f"  Location: {latest}")
        print(f"\nYou can access this from any Colab session by mounting Drive!")
else:
    print("⚠️  No checkpoints found in Google Drive")

## 9. Test the Checkpoint

Let's test text generation with the trained checkpoint (from Google Drive).

In [ ]:
# Find the latest checkpoint in Google Drive
drive_checkpoint_dir = Path("/content/drive/MyDrive/colab/janogpt_checkpoints/colab_tpu")
checkpoints = sorted(drive_checkpoint_dir.glob("step_*"))

if checkpoints:
    latest_checkpoint = checkpoints[-1]
    print(f"Testing with checkpoint from Drive: {latest_checkpoint.name}")

    # Generate text
    !python scripts/generate.py \
        --checkpoint {latest_checkpoint} \
        --prompt "Once upon a time" \
        --max_tokens 100 \
        --temperature 0.8
else:
    print("No checkpoint found in Google Drive")

## 10. Download Checkpoint as Zip (Optional)

If you want to download a checkpoint to your local machine, create a zip file.

In [ ]:
# Create zip of latest checkpoint from Google Drive
import shutil

if checkpoints:
    latest_checkpoint = checkpoints[-1]
    zip_name = f"{latest_checkpoint.name}_tpu"
    zip_path = f"/content/{zip_name}.zip"

    print(f"Creating archive: {zip_name}.zip")
    shutil.make_archive(
        f"/content/{zip_name}",
        'zip',
        drive_checkpoint_dir,
        latest_checkpoint.name
    )

    zip_size = Path(zip_path).stat().st_size / 1e6
    print(f"✓ Archive created: {zip_path} ({zip_size:.1f} MB)")
    print(f"\nDownload from the Files panel (left sidebar) or run:")
    print(f"  from google.colab import files")
    print(f"  files.download('{zip_path}')")
else:
    print("No checkpoint to archive")

## Summary

**What we accomplished:**
1. ✅ Set up JanoGPT on Colab with TPU v5e-1
2. ✅ Configured WandB tracking
3. ✅ Downloaded OpenWebText dataset to Google Drive (~20GB)
4. ✅ Trained GPT-2 124M for 100 steps on TPU
5. ✅ Saved checkpoints every 50 steps
6. ✅ **Uploaded checkpoints to Google Drive**
7. ✅ Tested text generation

**Memory optimization (OSC formula):**
- Base overhead: `40 × 0.124B params = ~5GB` (model + gradients + optimizer)
- Activations: `~34MB per sequence` (empirically measured for GPT-2 124M, seq_len=1024)

**Conservative TPU configuration (16GB v5e):**
- micro_batch=4, accum=128 → ~524K tokens/step
- Estimated memory: ~5.1GB (~11GB headroom)
- Safe configuration that avoids OOM on Colab TPU

**Checkpoint persistence:**
- ✅ Saved to: `/content/drive/MyDrive/colab/janogpt_checkpoints/colab_tpu/`
- ✅ Accessible from any Colab session (mount Drive)
- ✅ Survives session termination

**TPU Performance:**
- TPU v5e-1: ~8-16K tokens/sec (8-10× faster than single T4 GPU)
- Optimized for large batch sizes with gradient accumulation
- Conservative memory settings ensure stable training

**Resume training from Drive:**
```python
# In a new session:
drive.mount('/content/drive')
checkpoint = "/content/drive/MyDrive/colab/janogpt_checkpoints/colab_tpu/step_100"
# Update config with resume_from_checkpoint and max_steps
```